In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate
import torch

In [ ]:
df = pd.read_csv(r"C:\Users\shahj\Downloads\Dataset-SA-Augmented-40000.csv")
df = df[["Review","Sentiment"]].dropna()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v:k for k,v in label2id.items()}
df["label"] = df["Sentiment"].map(label2id)

train_df = df.sample(frac=0.8, random_state=42)
valid_df = df.drop(train_df.index)

train_ds = Dataset.from_pandas(train_df[["Review","label"]])
valid_ds = Dataset.from_pandas(valid_df[["Review","label"]])

In [ ]:
INDICBERT = "ai4bharat/indic-bert"
XLMR = "xlm-roberta-base"

In [ ]:
MAX_LEN = 128

def preprocess(tokenizer):
    def fn(examples):
        return tokenizer(
            examples["Review"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        )
    return fn

In [ ]:
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = metric_acc.compute(predictions=preds, references=labels)
    f1_macro = metric_f1.compute(predictions=preds, references=labels, average="macro")

    return {"accuracy": acc["accuracy"], "f1_macro": f1_macro["f1"]}

In [ ]:
def build_preprocess_fn(tokenizer, max_length=128, remove_token_type_ids=True):
    def fn(batch):
        enc = tokenizer(
            batch["Review"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )
        # remove token_type_ids for IndicBERT
        if remove_token_type_ids and "token_type_ids" in enc:
            del enc["token_type_ids"]
        return enc
    return fn


def train_model(model_name, output_dir, remove_token_type_ids=True, use_auth_token=None):
    print(f"\nTraining {model_name}\n")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=use_auth_token)
    preprocess_fn = build_preprocess_fn(tokenizer, remove_token_type_ids=remove_token_type_ids)

    # Tokenize datasets
    train_tok = train_ds.map(preprocess_fn, batched=True)
    valid_tok = valid_ds.map(preprocess_fn, batched=True)

    # Rename label column → labels
    train_tok = train_tok.rename_column("label", "labels")
    valid_tok = valid_tok.rename_column("label", "labels")

    # Set PyTorch format
    train_tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    valid_tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
        use_auth_token=use_auth_token
    )

    # Correct training arguments for Transformers 4.35.2
    args = TrainingArguments(
        output_dir=output_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),   # FP16 on RTX 4060
        dataloader_num_workers=0,
        dataloader_pin_memory=False
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=valid_tok,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    print("Saved:", output_dir)
    return trainer

In [ ]:
from huggingface_hub import login

login(token="YOUR_TOKEN_HERE")  # Replace with your Hugging Face token


In [ ]:
trainer_indicbert = train_model(INDICBERT, "indicbert-finetuned")

In [ ]:
trainer_xlmr = train_model(XLMR, "xlmr-finetuned")

In [ ]:
print("IndicBERT Results:")
trainer_indicbert.evaluate()

print("XLM-RoBERTa Results:")
trainer_xlmr.evaluate()

In [ ]:
pip install accelerate==0.21.0

In [ ]:
print("IndicBERT Results:")
trainer_indicbert.evaluate()

In [ ]:
print("XLM-RoBERTa Results:")
trainer_xlmr.evaluate()

In [ ]:
!pip install shap 

In [ ]:
import shap
import torch 